In [0]:
# ============================================================================
# CONFIGURATION: Set the target catalog.schema for all metric views
# ============================================================================
# Change these widgets to deploy the conformed dimension MVs to any schema.
# All subsequent SQL cells use these values via ${catalog} and ${schema}.
# ============================================================================

dbutils.widgets.text("catalog", "home_dipankar_kushari", "Target Catalog")
dbutils.widgets.text("schema", "metric_view", "Target Schema")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Target: {catalog}.{schema}")
print(f"All metric views will be created in: {catalog}.{schema}")

Target: home_dipankar_kushari.metric_view
All metric views will be created in: home_dipankar_kushari.metric_view


In [0]:
%sql
-- ============================================================================
-- CONFORMED DIMENSIONS: Patterns That Work Today + Known Limitations
-- ============================================================================
--
-- This notebook demonstrates conformed dimension patterns using samples.tpch
-- and addresses three requirements:
--
-- 1. NO REPEATED DIMENSION COLUMNS
--    Each dimension MV owns ONLY its own attributes. Period.
--    region_name lives ONLY in region_dim_mv.
--    nation_name lives ONLY in nation_dim_mv.
--    Consumer MVs access them via dot-path notation.
--
-- 2. BRIDGE TABLE PATTERN
--    When two facts share multiple conformed dimensions (supplier AND part),
--    a bridge MV holds the valid (supplier_key, part_key) combinations.
--    ⚠️ LIMITATION: A bridge that cross-joins two dim MVs (ON "1=1") does NOT
--    work today — fan-out protection blocks it. The bridge must be sourced
--    from actual data (e.g., DISTINCT keys from a fact table).
--
-- 3. JOINING FACTS THROUGH CONFORMED DIMENSIONS
--    Using the dimension MV as source with one_to_many joins to both facts.
--
-- ============================================================================
-- SCHEMA:    :catalog.:schema  (configurable via widgets above)
-- SOURCE:    samples.tpch
-- REQUIRES:  DBR 18.1+ (for cardinality: one_to_many, wildcard: source.*)
-- ============================================================================

SELECT :catalog || '.' || :schema AS target_schema, 'Conformed Dimensions Example' AS title

target_schema,title
home_dipankar_kushari.metric_view,Conformed Dimensions Example


In [0]:
%sql
drop schema if exists IDENTIFIER(:catalog || '.' || :schema) cascade;
create schema if not exists IDENTIFIER(:catalog || '.' || :schema);

In [0]:
%sql
use catalog IDENTIFIER(:catalog);
use schema IDENTIFIER(:schema);

In [0]:
%sql
select current_catalog();

current_catalog()
home_dipankar_kushari


In [0]:
%sql
select current_schema();

current_schema()
metric_view


### Pattern 1 &mdash; Conformed Dimensions (No Repeated Columns)
<p style="font-family:-apple-system,sans-serif;color:#5F747E;font-size:13px;margin-bottom:28px;">Each dimension MV owns ONLY its own attributes &bull; Consumers access related attributes via dot-path notation through joins</p>

<div style="font-family:-apple-system,sans-serif;max-width:1200px;margin:0 auto;">

<style>
.cd2-row{display:flex;justify-content:center;align-items:stretch;gap:0;}
.cd2-card{width:220px;min-height:200px;background:#fff;border-radius:6px;box-shadow:0 1px 3px rgba(27,58,75,.08),0 0 0 1px rgba(27,58,75,.10);display:flex;flex-direction:column;align-items:flex-start;padding:14px 13px 11px;position:relative;flex-shrink:0;overflow:hidden;}
.cd2-card::before{content:'';position:absolute;top:0;left:0;width:100%;height:4px;border-radius:6px 6px 0 0;}
.cd2-card.dim::before{background:#1B3A4B;}
.cd2-card.dim-alt::before{background:#FF3621;}
.cd2-card.dim-indep::before{background:#6A1B9A;}
.cd2-card .pn{font-size:9px;font-weight:600;text-transform:uppercase;letter-spacing:.7px;color:#8DA6B0;margin-top:6px;margin-bottom:2px;}
.cd2-card .pt{font-size:13.5px;font-weight:600;color:#1B3A4B;margin-bottom:8px;line-height:1.3;}
.cd2-card .pb{font-size:11.5px;color:#3D5868;line-height:1.6;flex-grow:1;width:100%;}
.cd2-card .pb ul{margin:0;padding:0;list-style:none;}
.cd2-card .pb li{margin-bottom:5px;padding-left:12px;position:relative;}
.cd2-card .pb li::before{content:'\203A';position:absolute;left:0;color:#FF3621;font-weight:600;line-height:1.4;}
.cd2-card .pb code{font-family:monospace;font-size:10px;background:#F0EDE8;color:#1B3A4B;padding:1px 4px;border-radius:3px;}
.cd2-card .pb strong{font-weight:600;color:#1B3A4B;}
.cd2-conn{display:flex;align-items:stretch;justify-content:center;width:70px;flex-shrink:0;}
.cd2-sep{display:flex;align-items:center;justify-content:center;width:60px;flex-shrink:0;}
.cd2-sep-line{width:1px;height:80%;background:repeating-linear-gradient(to bottom,#C4D4DB 0,#C4D4DB 4px,transparent 4px,transparent 8px);}
</style>

<!-- FK Chain: region → nation → supplier -->
<p style="font-size:10px;font-weight:600;text-transform:uppercase;letter-spacing:.8px;color:#1B3A4B;margin-bottom:10px;text-align:center;">FK Chain (geographic hierarchy)</p>

<div class="cd2-row">

  <div class="cd2-card dim">
    <div class="pn">Dimension</div>
    <div class="pt">🌍 region_dim_mv</div>
    <div class="pb"><ul>
      <li><strong>PK:</strong> <code>region_key</code></li>
      <li><strong>Owns:</strong> <code>region_name</code></li>
      <li>Source: <code>samples.tpch.region</code></li>
    </ul></div>
  </div>

  <!-- Cornered arrow: region_key PK → nation's FK region_key -->
  <div class="cd2-conn" style="position:relative;">
    <div style="position:absolute;top:34%;left:0;width:50%;height:1.5px;background:#8DA6B0;"></div>
    <div style="position:absolute;top:34%;left:50%;width:1.5px;height:23%;background:#8DA6B0;"></div>
    <div style="position:absolute;top:57%;left:50%;width:calc(50% - 5px);height:1.5px;background:#8DA6B0;"></div>
    <div style="position:absolute;top:57%;right:0;transform:translateY(-50%);width:0;height:0;border-left:5px solid #8DA6B0;border-top:3px solid transparent;border-bottom:3px solid transparent;"></div>
    <div style="position:absolute;top:28%;left:50%;transform:translateX(-50%);font-size:8.5px;color:#5F747E;font-weight:600;white-space:nowrap;font-family:-apple-system,sans-serif;">region_key</div>
  </div>

  <div class="cd2-card dim-alt">
    <div class="pn">Dimension</div>
    <div class="pt">🇺🇳 nation_dim_mv</div>
    <div class="pb"><ul>
      <li><strong>PK:</strong> <code>nation_key</code></li>
      <li><strong>Owns:</strong> <code>nation_name</code></li>
      <li><strong>FK:</strong> <code>region_key</code> &rarr; region</li>
      <li>Does <strong>NOT</strong> redeclare <code>region_name</code></li>
    </ul></div>
  </div>

  <!-- Cornered arrow: nation_key PK → supplier's FK nation_key -->
  <div class="cd2-conn" style="position:relative;">
    <div style="position:absolute;top:34%;left:0;width:50%;height:1.5px;background:#8DA6B0;"></div>
    <div style="position:absolute;top:34%;left:50%;width:1.5px;height:23%;background:#8DA6B0;"></div>
    <div style="position:absolute;top:57%;left:50%;width:calc(50% - 5px);height:1.5px;background:#8DA6B0;"></div>
    <div style="position:absolute;top:57%;right:0;transform:translateY(-50%);width:0;height:0;border-left:5px solid #8DA6B0;border-top:3px solid transparent;border-bottom:3px solid transparent;"></div>
    <div style="position:absolute;top:28%;left:50%;transform:translateX(-50%);font-size:8.5px;color:#5F747E;font-weight:600;white-space:nowrap;font-family:-apple-system,sans-serif;">nation_key</div>
  </div>

  <div class="cd2-card dim">
    <div class="pn">Dimension</div>
    <div class="pt">🏭 supplier_dim_mv</div>
    <div class="pb"><ul>
      <li><strong>PK:</strong> <code>supplier_key</code></li>
      <li><strong>Owns:</strong> <code>supplier_name</code></li>
      <li><strong>FK:</strong> <code>nation_key</code> &rarr; nation</li>
      <li>Does <strong>NOT</strong> redeclare nation/region</li>
    </ul></div>
  </div>

  <!-- Visual separator (dashed) -->
  <div class="cd2-sep"><div class="cd2-sep-line"></div></div>

  <!-- Independent dimension -->
  <div class="cd2-card dim-indep">
    <div class="pn">Independent Dimension</div>
    <div class="pt">📦 part_dim_mv</div>
    <div class="pb"><ul>
      <li><strong>PK:</strong> <code>part_key</code></li>
      <li><strong>Owns:</strong> <code>part_name</code>, <code>brand</code>, <code>part_type</code></li>
      <li>No FK to any other dimension</li>
      <li>Source: <code>samples.tpch.part</code></li>
    </ul></div>
  </div>

</div>

<p style="text-align:center;color:#5F747E;font-size:11px;margin-top:16px;"><em>Key Rule: To get region_name for a supplier, traverse the dot-path chain &rarr; <code>supplier_dim_mv.nation.region.region_name</code></em></p>

</div>

In [0]:
%sql
-- ============================================================================
-- PATTERN 1: CONFORMED DIMENSIONS — NO REPEATED COLUMNS
-- ============================================================================
--
-- KEY RULE: Each dimension MV declares ONLY the attributes it OWNS.
--           It does NOT re-declare attributes from other dimensions.
--
--   region_dim_mv  → owns: region_key, region_name
--   nation_dim_mv  → owns: nation_key, nation_name (NOT region_name!)
--   supplier_dim_mv → owns: supplier_key, supplier_name (NOT nation_name, NOT region_name!)
--   part_dim_mv    → owns: part_key, part_name, brand, type
--
-- To get region_name when querying a supplier, you traverse the chain:
--   supplier_dim_mv.nation.region.region_name  (dot-path through joins)
--
-- This is the TRUE conformed dimension pattern from the document:
--   "A fact view exposes its measures plus the foreign keys that relate it
--    to dimensions — never the descriptive attributes themselves."
-- ============================================================================

-- ┌──────────────────────────────────────────────┐
-- │  region_dim_mv — owns ONLY region attributes │
-- └──────────────────────────────────────────────┘
CREATE OR REPLACE VIEW region_dim_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: "Conformed dimension: Geographic regions. Owns region_key and region_name ONLY."
source: samples.tpch.region
dimensions:
  - name: region_key
    expr: r_regionkey
    comment: "PK for region"
  - name: region_name
    expr: r_name
    comment: "Region name (AFRICA, AMERICA, ASIA, EUROPE, MIDDLE EAST)"
    display_name: "Region"
    synonyms:
      - "continent"
      - "geographic region"
$$

In [0]:
%sql
-- ┌──────────────────────────────────────────────────────────────────────┐
-- │  nation_dim_mv — owns ONLY nation attributes + FK to region         │
-- │  Does NOT re-declare region_name! That belongs to region_dim_mv.    │
-- └──────────────────────────────────────────────────────────────────────┘
CREATE OR REPLACE VIEW nation_dim_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: "Conformed dimension: Nations. Owns nation_key, nation_name, and FK region_key. Does NOT repeat region_name."
source: samples.tpch.nation
dimensions:
  - name: nation_key
    expr: n_nationkey
    comment: "PK for nation"
  - name: nation_name
    expr: n_name
    comment: "Country name"
    display_name: "Nation"
    synonyms:
      - "country"
      - "country name"
  - name: region_key
    expr: n_regionkey
    comment: "FK to region_dim_mv — use this to join, NOT to re-declare region_name here"
$$

In [0]:
%sql
-- ┌──────────────────────────────────────────────────────────────────────┐
-- │  supplier_dim_mv — owns ONLY supplier attributes + FK to nation     │
-- │  Does NOT re-declare nation_name or region_name!                     │
-- └──────────────────────────────────────────────────────────────────────┘
CREATE OR REPLACE VIEW supplier_dim_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: "Conformed dimension: Suppliers. Owns supplier_key, supplier_name, and FK nation_key. Does NOT repeat nation/region attributes."
source: samples.tpch.supplier
dimensions:
  - name: supplier_key
    expr: s_suppkey
    comment: "PK for supplier"
  - name: supplier_name
    expr: s_name
    comment: "Supplier name"
    display_name: "Supplier"
    synonyms:
      - "vendor"
  - name: nation_key
    expr: s_nationkey
    comment: "FK to nation_dim_mv — NOT re-declaring nation_name here"
$$

In [0]:
%sql
-- ┌──────────────────────────────────────────────────────────────────────┐
-- │  part_dim_mv — owns ONLY part attributes                            │
-- └──────────────────────────────────────────────────────────────────────┘
CREATE OR REPLACE VIEW part_dim_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: "Conformed dimension: Parts/products. Owns part attributes only."
source: samples.tpch.part
dimensions:
  - name: part_key
    expr: p_partkey
    comment: "PK for part"
  - name: part_name
    expr: p_name
    comment: "Part name"
    display_name: "Part"
  - name: brand
    expr: p_brand
    comment: "Part brand"
    display_name: "Brand"
  - name: part_type
    expr: p_type
    comment: "Part type"
    display_name: "Type"
$$

### Fact Metric Views &mdash; Measures + Foreign Keys Only
<p style="font-family:-apple-system,sans-serif;color:#5F747E;font-size:13px;margin-bottom:28px;">Facts expose aggregations and FK references &bull; They NEVER re-declare descriptive dimension attributes</p>

<div style="font-family:-apple-system,sans-serif;max-width:900px;margin:0 auto;">

<style>
.fct-row{display:flex;justify-content:center;align-items:stretch;gap:32px;}
.fct-card{width:280px;min-height:200px;background:#fff;border-radius:6px;box-shadow:0 1px 3px rgba(27,58,75,.08),0 0 0 1px rgba(27,58,75,.10);display:flex;flex-direction:column;align-items:flex-start;padding:14px 13px 11px;position:relative;flex-shrink:0;overflow:hidden;}
.fct-card::before{content:'';position:absolute;top:0;left:0;width:100%;height:4px;border-radius:6px 6px 0 0;}
.fct-card.f1::before{background:#2E7D32;}
.fct-card.f2::before{background:#E65100;}
.fct-card .pn{font-size:9px;font-weight:600;text-transform:uppercase;letter-spacing:.7px;color:#8DA6B0;margin-top:6px;margin-bottom:2px;}
.fct-card .pt{font-size:13.5px;font-weight:600;color:#1B3A4B;margin-bottom:8px;line-height:1.3;}
.fct-card .pb{font-size:11.5px;color:#3D5868;line-height:1.6;flex-grow:1;width:100%;}
.fct-card .pb ul{margin:0;padding:0;list-style:none;}
.fct-card .pb li{margin-bottom:5px;padding-left:12px;position:relative;}
.fct-card .pb li::before{content:'\203A';position:absolute;left:0;color:#2E7D32;font-weight:600;line-height:1.4;}
.fct-card .pb code{font-family:monospace;font-size:10px;background:#F0EDE8;color:#1B3A4B;padding:1px 4px;border-radius:3px;}
.fct-card .pb strong{font-weight:600;color:#1B3A4B;}
</style>

<div class="fct-row">

  <div class="fct-card f1">
    <div class="pn">Fact</div>
    <div class="pt">📊 lineitem_fact_mv</div>
    <div class="pb"><ul>
      <li><strong>FK:</strong> <code>l_suppkey</code> &rarr; supplier_dim_mv</li>
      <li><strong>FK:</strong> <code>l_partkey</code> &rarr; part_dim_mv</li>
      <li><strong>Measure:</strong> <code>total_quantity</code> &mdash; SUM(l_quantity)</li>
      <li><strong>Measure:</strong> <code>total_revenue</code> &mdash; SUM(price*(1-disc))</li>
      <li>Source: <code>samples.tpch.lineitem</code></li>
    </ul></div>
  </div>

  <div class="fct-card f2">
    <div class="pn">Fact</div>
    <div class="pt">📦 partsupp_fact_mv</div>
    <div class="pb"><ul>
      <li><strong>FK:</strong> <code>ps_suppkey</code> &rarr; supplier_dim_mv</li>
      <li><strong>FK:</strong> <code>ps_partkey</code> &rarr; part_dim_mv</li>
      <li><strong>Measure:</strong> <code>total_availqty</code> &mdash; SUM(ps_availqty)</li>
      <li><strong>Measure:</strong> <code>total_supplycost</code> &mdash; SUM(ps_supplycost)</li>
      <li>Source: <code>samples.tpch.partsupp</code></li>
    </ul></div>
  </div>

</div>

<p style="text-align:center;color:#5F747E;font-size:11px;margin-top:16px;"><em>Both facts share TWO conformed dimensions: supplier (via suppkey) and part (via partkey)</em></p>

</div>

In [0]:
%sql
-- ============================================================================
-- FACT METRIC VIEWS — measures + foreign keys ONLY
-- ============================================================================
-- Facts expose ONLY:
--   • Their own measures (aggregations)
--   • Foreign keys to relate to dimensions
-- They NEVER re-declare descriptive attributes (no supplier_name, no nation_name)
-- ============================================================================

-- ┌──────────────────────────────────────────────────────────────────────┐
-- │  lineitem_fact_mv — FK: l_suppkey, l_partkey                         │
-- └──────────────────────────────────────────────────────────────────────┘
CREATE OR REPLACE VIEW lineitem_fact_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: "Fact: Order line items. Measures + FKs only. No descriptive dimension attributes."
source: samples.tpch.lineitem
dimensions:
  - name: l_suppkey
    expr: l_suppkey
    comment: "FK to supplier_dim_mv"
  - name: l_partkey
    expr: l_partkey
    comment: "FK to part_dim_mv"
measures:
  - name: total_quantity
    expr: SUM(l_quantity)
    comment: "Total units ordered"
    display_name: "Qty Ordered"
  - name: total_revenue
    expr: SUM(l_extendedprice * (1 - l_discount))
    comment: "Net revenue after discount"
    display_name: "Revenue"
$$

In [0]:
%sql
-- ┌──────────────────────────────────────────────────────────────────────┐
-- │  partsupp_fact_mv — FK: ps_suppkey, ps_partkey                       │
-- └──────────────────────────────────────────────────────────────────────┘
CREATE OR REPLACE VIEW partsupp_fact_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: "Fact: Part-supplier inventory. Measures + FKs only. No descriptive dimension attributes."
source: samples.tpch.partsupp
dimensions:
  - name: ps_suppkey
    expr: ps_suppkey
    comment: "FK to supplier_dim_mv"
  - name: ps_partkey
    expr: ps_partkey
    comment: "FK to part_dim_mv"
measures:
  - name: total_availqty
    expr: SUM(ps_availqty)
    comment: "Total parts available in stock"
    display_name: "Available Qty"
  - name: total_supplycost
    expr: SUM(ps_supplycost)
    comment: "Total supply cost"
    display_name: "Supply Cost"
$$

### Pattern 2 &mdash; Bridge Table for Multiple Conformed Dimensions
<p style="font-family:-apple-system,sans-serif;color:#5F747E;font-size:13px;margin-bottom:28px;">When two facts share TWO keys, joining on only one causes fan-out &bull; A bridge holds valid (supplier, part) combinations</p>

<div style="font-family:-apple-system,sans-serif;max-width:1100px;margin:0 auto;">

<style>
.br-row{display:flex;justify-content:center;align-items:center;gap:0;}
.br-card{width:200px;min-height:140px;background:#fff;border-radius:6px;box-shadow:0 1px 3px rgba(27,58,75,.08),0 0 0 1px rgba(27,58,75,.10);display:flex;flex-direction:column;align-items:flex-start;padding:12px 11px 10px;position:relative;flex-shrink:0;overflow:hidden;}
.br-card::before{content:'';position:absolute;top:0;left:0;width:100%;height:4px;border-radius:6px 6px 0 0;}
.br-card.fact::before{background:#2E7D32;}
.br-card.bridge::before{background:#6A1B9A;}
.br-card .pn{font-size:9px;font-weight:600;text-transform:uppercase;letter-spacing:.7px;color:#8DA6B0;margin-top:4px;margin-bottom:2px;}
.br-card .pt{font-size:12.5px;font-weight:600;color:#1B3A4B;margin-bottom:6px;line-height:1.3;}
.br-card .pb{font-size:11px;color:#3D5868;line-height:1.5;flex-grow:1;width:100%;}
.br-card .pb code{font-family:monospace;font-size:10px;background:#F0EDE8;color:#1B3A4B;padding:1px 4px;border-radius:3px;}
.br-card .pb strong{font-weight:600;color:#1B3A4B;}
.br-arrow{display:flex;align-items:center;justify-content:center;width:60px;flex-shrink:0;}
.br-arrow-b{position:relative;width:36px;height:2px;background:#6A1B9A;border-radius:1px;}
.br-arrow-b.right::after{content:'';position:absolute;right:-7px;top:-4px;border-left:8px solid #6A1B9A;border-top:5px solid transparent;border-bottom:5px solid transparent;}
.br-arrow-b.left::after{content:'';position:absolute;left:-7px;top:-4px;border-right:8px solid #6A1B9A;border-top:5px solid transparent;border-bottom:5px solid transparent;}
.br-label{font-size:9px;color:#6A1B9A;font-weight:600;text-align:center;margin-top:4px;}
</style>

<div class="br-row">

  <div class="br-card fact">
    <div class="pn">Fact</div>
    <div class="pt">📊 lineitem</div>
    <div class="pb">
      <code>l_suppkey</code> + <code>l_partkey</code><br/>
      <strong>Measures:</strong> qty, revenue
    </div>
  </div>

  <div class="br-arrow">
    <div style="text-align:center;">
      <div class="br-arrow-b left"></div>
      <div class="br-label">BOTH keys</div>
    </div>
  </div>

  <div class="br-card bridge">
    <div class="pn">Bridge</div>
    <div class="pt">🔗 supplier_part_bridge</div>
    <div class="pb">
      <code>supplier_key</code><br/>
      <code>part_key</code><br/>
      <strong>Valid combinations only</strong>
    </div>
  </div>

  <div class="br-arrow">
    <div style="text-align:center;">
      <div class="br-arrow-b right"></div>
      <div class="br-label">BOTH keys</div>
    </div>
  </div>

  <div class="br-card fact">
    <div class="pn">Fact</div>
    <div class="pt">📦 partsupp</div>
    <div class="pb">
      <code>ps_suppkey</code> + <code>ps_partkey</code><br/>
      <strong>Measures:</strong> qty, cost
    </div>
  </div>

</div>

<p style="text-align:center;color:#5F747E;font-size:11px;margin-top:16px;"><em>⚠️ Joining on only supplier_key &rarr; 80&times; fan-out &bull; Bridge on BOTH keys &rarr; correct totals</em></p>

</div>

In [0]:
%sql
-- ============================================================================
-- PATTERN 2: BRIDGE TABLE — for multiple conformed dimensions
-- ============================================================================
--
-- PROBLEM: lineitem and partsupp share TWO conformed dimensions:
--   • supplier (l_suppkey / ps_suppkey)
--   • part (l_partkey / ps_partkey)
--
-- If you join the two facts through only ONE of these keys, you get fan-out
-- because each supplier has many parts and vice versa.
--
-- SOLUTION: A bridge MV that holds the valid (supplier_key, part_key) 
-- combinations. Each fact relates to the bridge on BOTH keys, preventing fan-out.
--
-- In TPC-H, the partsupp table IS the natural bridge — it contains exactly
-- the valid (supplier, part) pairs. The bridge is a DISTINCT projection of
-- those keys.
--
-- ⚠️ KNOWN LIMITATION:
--    You CANNOT create a bridge by cross-joining two dim MVs (ON "1=1")
--    in the joins: section. Fan-out protection blocks this.
--    The bridge MUST be sourced from actual data with real key combinations.
-- ============================================================================

CREATE OR REPLACE VIEW supplier_part_bridge_dim_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Bridge dimension: Valid (supplier, part) combinations.
  Sourced from partsupp which enumerates exactly which suppliers supply which parts.
  Used to join lineitem and partsupp facts on BOTH conformed keys without fan-out.
  NOTE: Cannot be built by cross-joining dim MVs (fan-out protection blocks ON 1=1).
source: >
  SELECT DISTINCT ps_suppkey AS supplier_key, ps_partkey AS part_key
  FROM samples.tpch.partsupp
dimensions:
  - name: supplier_key
    expr: supplier_key
    comment: "FK to supplier_dim_mv"
  - name: part_key
    expr: part_key
    comment: "FK to part_dim_mv"
$$

### Pattern 2b &mdash; DRY Bridge (Inherit Metadata from Dim MVs)
<p style="font-family:-apple-system,sans-serif;color:#5F747E;font-size:13px;margin-bottom:28px;">Cross-join in SOURCE bypasses fan-out protection &bull; Re-join dim MVs with <code>rely: at_most_one_match</code> to inherit display_names/synonyms</p>

<div style="font-family:-apple-system,sans-serif;max-width:1000px;margin:0 auto;">

<style>
.dry-row{display:flex;justify-content:center;align-items:stretch;gap:0;}
.dry-card{width:240px;min-height:160px;background:#fff;border-radius:6px;box-shadow:0 1px 3px rgba(27,58,75,.08),0 0 0 1px rgba(27,58,75,.10);display:flex;flex-direction:column;align-items:flex-start;padding:12px 11px 10px;position:relative;flex-shrink:0;overflow:hidden;}
.dry-card::before{content:'';position:absolute;top:0;left:0;width:100%;height:4px;border-radius:6px 6px 0 0;}
.dry-card.step1::before{background:#1B3A4B;}
.dry-card.step2::before{background:#FF3621;}
.dry-card.step3::before{background:#6A1B9A;}
.dry-card .pn{font-size:9px;font-weight:600;text-transform:uppercase;letter-spacing:.7px;color:#8DA6B0;margin-top:4px;margin-bottom:2px;}
.dry-card .pt{font-size:13px;font-weight:600;color:#1B3A4B;margin-bottom:6px;line-height:1.3;}
.dry-card .pb{font-size:11px;color:#3D5868;line-height:1.6;flex-grow:1;width:100%;}
.dry-card .pb ul{margin:0;padding:0;list-style:none;}
.dry-card .pb li{margin-bottom:4px;padding-left:12px;position:relative;}
.dry-card .pb li::before{content:'\203A';position:absolute;left:0;color:#FF3621;font-weight:600;line-height:1.4;}
.dry-card .pb code{font-family:monospace;font-size:10px;background:#F0EDE8;color:#1B3A4B;padding:1px 4px;border-radius:3px;}
.dry-card .pb strong{font-weight:600;color:#1B3A4B;}
.dry-arr{display:flex;align-items:center;justify-content:center;width:40px;flex-shrink:0;}
.dry-arr-b{position:relative;width:28px;height:2px;background:#C4D4DB;border-radius:1px;}
.dry-arr-b::after{content:'';position:absolute;right:-7px;top:-4px;border-left:8px solid #C4D4DB;border-top:5px solid transparent;border-bottom:5px solid transparent;}
</style>

<div class="dry-row">

  <div class="dry-card step1">
    <div class="pn">Step 1 &mdash; Source</div>
    <div class="pt">✖ CROSS JOIN in source:</div>
    <div class="pb"><ul>
      <li><code>nation_dim_mv</code> CROSS JOIN <code>supplier_dim_mv</code></li>
      <li>Produces all (nation, supplier) pairs</li>
      <li>Bypasses fan-out protection</li>
    </ul></div>
  </div>

  <div class="dry-arr"><div class="dry-arr-b"></div></div>

  <div class="dry-card step2">
    <div class="pn">Step 2 &mdash; Joins</div>
    <div class="pt">🔄 Re-attach dim MVs</div>
    <div class="pb"><ul>
      <li>Join <code>nation_dim_mv</code> (many_to_one)</li>
      <li>Join <code>supplier_dim_mv</code> (many_to_one)</li>
      <li><code>rely: at_most_one_match: true</code></li>
    </ul></div>
  </div>

  <div class="dry-arr"><div class="dry-arr-b"></div></div>

  <div class="dry-card step3">
    <div class="pn">Step 3 &mdash; Dimensions</div>
    <div class="pt">✨ Wildcard EXCEPT</div>
    <div class="pb"><ul>
      <li><code>nation.* EXCEPT (nation_key)</code></li>
      <li><code>supplier.* EXCEPT (supplier_key)</code></li>
      <li>Inherits ALL metadata (display_name, synonyms)</li>
      <li>Zero redeclaration &mdash; truly DRY</li>
    </ul></div>
  </div>

</div>

<p style="text-align:center;color:#5F747E;font-size:11px;margin-top:16px;"><em>Requires DBR 18.1+ for rely: at_most_one_match and wildcard EXCEPT syntax</em></p>

</div>

In [0]:
%sql
-- ============================================================================
-- PATTERN 2b: BRIDGE WITH DRY METADATA
-- ============================================================================
--
-- PROBLEM: You want a bridge that inherits display_names/synonyms from
--          the conformed dim MVs without redefining them. But a cross-join
--          in the joins: section fails (fan-out protection blocks ON "1=1").
--
-- WORKAROUND:
--   1. Put the CROSS JOIN in the SOURCE query (not in joins:)
--   2. Join BACK to the dim MVs as many_to_one (with rely: at_most_one_match)
--   3. Use wildcard (dim.* EXCEPT (key)) to inherit all metadata
--
-- This keeps things DRY — dimension definitions live in one place.
-- Downside: More expensive (cross-join in source). But it's the best
-- available approach today until Relationships + measures in SQL ship.
--
-- REQUIRES DBR 18.1+ for rely: at_most_one_match and wildcard EXCEPT
-- ============================================================================

-- DDL (run on DBR 18.1+):
--
CREATE OR REPLACE VIEW supplier_nation_bridge_dry_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Bridge: (nation, supplier) combinations with DRY metadata.
  Cross-join in SOURCE bypasses fan-out protection.
  Joins back to dim MVs to inherit display_names/synonyms.
source: |
  SELECT n.nation_key, s.supplier_key
  FROM nation_dim_mv n
  CROSS JOIN supplier_dim_mv s
joins:
  - name: nation
    source: nation_dim_mv
    on: "source.nation_key = nation.nation_key"
    rely:
      at_most_one_match: true
  - name: supplier
    source: supplier_dim_mv
    on: "source.supplier_key = supplier.supplier_key"
    rely:
      at_most_one_match: true
dimensions:
  - expr: nation.* EXCEPT (nation_key)
  - expr: supplier.* EXCEPT (supplier_key)
$$;
--
-- ============================================================================
-- HOW IT WORKS:
-- ============================================================================
--
-- 1. SOURCE does the cross-join → produces all (nation_key, supplier_key) pairs
--    This bypasses fan-out protection (only applies to joins: section)
--
-- 2. JOINS re-attach dim MVs as many_to_one lookups:
--    Each row matches exactly ONE nation and ONE supplier
--    → rely: at_most_one_match: true is truthful
--
-- 3. DIMENSIONS use wildcard EXCEPT to pull ALL metadata from dim MVs:
--    • nation.* EXCEPT (nation_key) → nation_name + display_name + synonyms
--    • supplier.* EXCEPT (supplier_key) → supplier_name + display_name + synonyms
--
-- Result: Bridge has full metadata from both dims, defined ZERO extra places.

SELECT 'Pattern 2b: DRY bridge via cross-join in SOURCE (requires DBR 18.1+)' AS pattern

pattern
Pattern 2b: DRY bridge via cross-join in SOURCE (requires DBR 18.1+)


### Pattern 3 &mdash; Joining Facts Through a Single Conformed Dimension
<p style="font-family:-apple-system,sans-serif;color:#5F747E;font-size:13px;margin-bottom:28px;">Dimension MV as source &bull; one_to_many joins to each fact &bull; MEASURE() aggregates each fact independently &mdash; no fan-out</p>

<div style="font-family:-apple-system,sans-serif;max-width:1000px;margin:0 auto;">

<style>
.p3-row{display:flex;justify-content:center;align-items:center;gap:0;}
.p3-card{width:210px;min-height:150px;background:#fff;border-radius:6px;box-shadow:0 1px 3px rgba(27,58,75,.08),0 0 0 1px rgba(27,58,75,.10);display:flex;flex-direction:column;align-items:flex-start;padding:12px 11px 10px;position:relative;flex-shrink:0;overflow:hidden;}
.p3-card::before{content:'';position:absolute;top:0;left:0;width:100%;height:4px;border-radius:6px 6px 0 0;}
.p3-card.src::before{background:#1B3A4B;}
.p3-card.dim::before{background:#FF3621;}
.p3-card.fct::before{background:#2E7D32;}
.p3-card .pn{font-size:9px;font-weight:600;text-transform:uppercase;letter-spacing:.7px;color:#8DA6B0;margin-top:4px;margin-bottom:2px;}
.p3-card .pt{font-size:12.5px;font-weight:600;color:#1B3A4B;margin-bottom:6px;line-height:1.3;}
.p3-card .pb{font-size:11px;color:#3D5868;line-height:1.5;flex-grow:1;width:100%;}
.p3-card .pb code{font-family:monospace;font-size:10px;background:#F0EDE8;color:#1B3A4B;padding:1px 4px;border-radius:3px;}
.p3-card .pb strong{font-weight:600;color:#1B3A4B;}
.p3-arr{display:flex;align-items:center;justify-content:center;width:44px;flex-shrink:0;}
.p3-arr-b{position:relative;width:28px;height:2px;background:#C4D4DB;border-radius:1px;}
.p3-arr-b.r::after{content:'';position:absolute;right:-7px;top:-4px;border-left:8px solid #C4D4DB;border-top:5px solid transparent;border-bottom:5px solid transparent;}
.p3-arr-b.l::after{content:'';position:absolute;left:-7px;top:-4px;border-right:8px solid #C4D4DB;border-top:5px solid transparent;border-bottom:5px solid transparent;}
.p3-lbl{font-size:8px;color:#5F747E;text-align:center;margin-top:3px;}
.p3-vert{display:flex;flex-direction:column;align-items:center;margin-top:16px;}
.p3-vert-line{width:2px;height:28px;background:#C4D4DB;border-radius:1px;position:relative;}
.p3-vert-line::after{content:'';position:absolute;bottom:-7px;left:-4px;border-top:8px solid #C4D4DB;border-left:5px solid transparent;border-right:5px solid transparent;}
.p3-vert-lbl{font-size:8px;color:#5F747E;margin-top:10px;margin-bottom:6px;}
</style>

<div class="p3-row">

  <div class="p3-card fct">
    <div class="pn">Fact (one_to_many)</div>
    <div class="pt">📊 lineitem</div>
    <div class="pb">
      <code>l_suppkey</code> &rarr; source<br/>
      <strong>Measures:</strong><br/>
      qty_ordered, order_revenue
    </div>
  </div>

  <div class="p3-arr">
    <div style="text-align:center;">
      <div class="p3-arr-b l"></div>
      <div class="p3-lbl">1:N</div>
    </div>
  </div>

  <div class="p3-card src">
    <div class="pn">Source (grain)</div>
    <div class="pt">🏭 supplier_dim_mv</div>
    <div class="pb">
      <code>supplier_key</code> (PK)<br/>
      <code>supplier_name</code><br/>
      <code>nation_key</code> (FK)
    </div>
  </div>

  <div class="p3-arr">
    <div style="text-align:center;">
      <div class="p3-arr-b r"></div>
      <div class="p3-lbl">1:N</div>
    </div>
  </div>

  <div class="p3-card fct">
    <div class="pn">Fact (one_to_many)</div>
    <div class="pt">📦 partsupp</div>
    <div class="pb">
      <code>ps_suppkey</code> &rarr; source<br/>
      <strong>Measures:</strong><br/>
      qty_in_stock, supply_cost
    </div>
  </div>

</div>

<!-- Vertical connector from supplier_dim_mv down to nation_dim_mv -->
<div class="p3-vert">
  <div class="p3-vert-line"></div>
  <div class="p3-vert-lbl">N:1 (many suppliers per nation)</div>
  <div class="p3-card dim" style="width:210px;min-height:auto;">
    <div class="pn">Dim (joined to source)</div>
    <div class="pt">🇺🇳 nation_dim_mv</div>
    <div class="pb">
      <code>nation.nation_name</code><br/>
      Joins on <code>source.nation_key</code><br/>
      Accessed via dot-path &mdash; <strong>NOT</strong> redeclared
    </div>
  </div>
</div>

<p style="text-align:center;color:#5F747E;font-size:11px;margin-top:16px;"><em>Query example: "Qty ordered vs. qty stocked, by nation" &mdash; each fact aggregates independently at its own grain</em></p>

</div>

In [0]:
%sql
-- ============================================================================
-- PATTERN 3: JOINING FACTS THROUGH A SINGLE CONFORMED DIMENSION
-- ============================================================================
--
-- When facts share ONE conformed dimension, use that dimension as source
-- with one_to_many joins to each fact. MEASURE() aggregates each fact at
-- its own grain independently — no fan-out.
--
-- Example: "Qty ordered vs. qty stocked, by nation"
--   supplier_dim_mv (source) → lineitem (one_to_many) + partsupp (one_to_many)
--   Then join nation_dim_mv for nation_name via dot-path.
--
-- KEY: nation_name from nation_dim_mv via dot-path — NOT redeclared!
-- ============================================================================

CREATE OR REPLACE VIEW orders_vs_supply_by_nation_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Multi-fact: Orders vs. Supply through shared supplier conformed dimension.
  nation_name accessed via dot-path from the canonical nation_dim_mv.
source: supplier_dim_mv
joins:
  - name: nation
    source: nation_dim_mv
    on: "nation.nation_key = source.nation_key"
  - name: lineitem
    source: samples.tpch.lineitem
    on: "lineitem.l_suppkey = source.supplier_key"
    cardinality: one_to_many
  - name: partsupp
    source: samples.tpch.partsupp
    on: "partsupp.ps_suppkey = source.supplier_key"
    cardinality: one_to_many
dimensions:
  - name: supplier_name
    expr: supplier_name
    comment: "From source (supplier_dim_mv) — owns this attribute"
  - name: nation_name
    expr: nation.nation_name
    comment: "From nation_dim_mv via dot-path — NOT redeclared"
measures:
  - name: qty_ordered
    expr: SUM(lineitem.l_quantity)
    display_name: "Qty Ordered"
  - name: order_revenue
    expr: SUM(lineitem.l_extendedprice * (1 - lineitem.l_discount))
    display_name: "Revenue"
  - name: qty_in_stock
    expr: SUM(partsupp.ps_availqty)
    display_name: "Qty In Stock"
  - name: supply_cost
    expr: SUM(partsupp.ps_supplycost)
    display_name: "Supply Cost"
$$

### Pattern 3b &mdash; Joining Facts Through Bridge (Multiple Conformed Dimensions)
<p style="font-family:-apple-system,sans-serif;color:#5F747E;font-size:13px;margin-bottom:28px;">Bridge MV as source &bull; Facts join on composite key (supplier + part) &bull; Dimension MVs provide descriptive attributes via dot-path</p>

<div style="font-family:-apple-system,sans-serif;max-width:1100px;margin:0 auto;">

<style>
.p3b-row{display:flex;justify-content:center;align-items:stretch;gap:0;}
.p3b-col{display:flex;flex-direction:column;align-items:center;gap:8px;}
.p3b-card{width:190px;min-height:120px;background:#fff;border-radius:6px;box-shadow:0 1px 3px rgba(27,58,75,.08),0 0 0 1px rgba(27,58,75,.10);display:flex;flex-direction:column;align-items:flex-start;padding:10px 10px 8px;position:relative;flex-shrink:0;overflow:hidden;}
.p3b-card::before{content:'';position:absolute;top:0;left:0;width:100%;height:4px;border-radius:6px 6px 0 0;}
.p3b-card.src::before{background:#6A1B9A;}
.p3b-card.dim::before{background:#1B3A4B;}
.p3b-card.fct::before{background:#2E7D32;}
.p3b-card .pn{font-size:8px;font-weight:600;text-transform:uppercase;letter-spacing:.6px;color:#8DA6B0;margin-top:4px;margin-bottom:2px;}
.p3b-card .pt{font-size:12px;font-weight:600;color:#1B3A4B;margin-bottom:4px;line-height:1.3;}
.p3b-card .pb{font-size:10.5px;color:#3D5868;line-height:1.5;flex-grow:1;width:100%;}
.p3b-card .pb code{font-family:monospace;font-size:9.5px;background:#F0EDE8;color:#1B3A4B;padding:1px 3px;border-radius:3px;}
.p3b-card .pb strong{font-weight:600;color:#1B3A4B;}
.p3b-arr{display:flex;align-items:center;justify-content:center;width:36px;flex-shrink:0;}
.p3b-arr-b{position:relative;width:22px;height:2px;background:#C4D4DB;border-radius:1px;}
.p3b-arr-b.r::after{content:'';position:absolute;right:-6px;top:-4px;border-left:7px solid #C4D4DB;border-top:5px solid transparent;border-bottom:5px solid transparent;}
.p3b-arr-b.l::after{content:'';position:absolute;left:-6px;top:-4px;border-right:7px solid #C4D4DB;border-top:5px solid transparent;border-bottom:5px solid transparent;}
</style>

<div class="p3b-row">

  <div class="p3b-col">
    <div class="p3b-card dim">
      <div class="pn">Dim</div>
      <div class="pt">🏭 supplier_dim_mv</div>
      <div class="pb"><code>supplier_name</code></div>
    </div>
    <div class="p3b-card dim">
      <div class="pn">Dim</div>
      <div class="pt">📦 part_dim_mv</div>
      <div class="pb"><code>brand</code></div>
    </div>
  </div>

  <div class="p3b-arr"><div class="p3b-arr-b l"></div></div>

  <div class="p3b-card src" style="align-self:center;min-height:180px;">
    <div class="pn">Source (grain)</div>
    <div class="pt">🔗 supplier_part_bridge</div>
    <div class="pb">
      <code>supplier_key</code><br/>
      <code>part_key</code><br/><br/>
      <strong>Valid (supplier, part)</strong><br/>
      combinations from data
    </div>
  </div>

  <div class="p3b-arr"><div class="p3b-arr-b r"></div></div>

  <div class="p3b-col">
    <div class="p3b-card fct">
      <div class="pn">Fact (one_to_many)</div>
      <div class="pt">📊 lineitem</div>
      <div class="pb">
        ON <code>suppkey</code> AND <code>partkey</code><br/>
        <strong>Measure:</strong> qty_ordered
      </div>
    </div>
    <div class="p3b-card fct">
      <div class="pn">Fact (one_to_many)</div>
      <div class="pt">💰 partsupp</div>
      <div class="pb">
        ON <code>suppkey</code> AND <code>partkey</code><br/>
        <strong>Measures:</strong> supply_cost, qty_in_stock
      </div>
    </div>
  </div>

</div>

<p style="text-align:center;color:#5F747E;font-size:11px;margin-top:16px;"><em>Bridge as direct source (not SQL expression) &rarr; avoids nested MV planning bug &bull; Facts join on FULL composite key</em></p>

</div>

In [0]:
%sql
-- ============================================================================
-- PATTERN 3b: JOINING FACTS THROUGH BRIDGE (multiple conformed dimensions)
-- ============================================================================
--
-- When facts share TWO conformed dimensions (supplier AND part), relating
-- through only one key re-introduces fan-out. Use the bridge instead.
--
-- The bridge (supplier_part_bridge_dim_mv) is the source.
-- Both facts join on the FULL composite key (supplier_key AND part_key).
-- Supplier and part dim MVs join to the bridge for descriptive attributes.
-- ============================================================================

CREATE OR REPLACE VIEW orders_vs_supply_by_supplier_part_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Multi-fact through BRIDGE: Orders vs. Supply by supplier AND part.
  Bridge prevents fan-out when joining on two conformed dimensions.
  Descriptive attributes from canonical dim MVs via dot-path.
source: supplier_part_bridge_dim_mv
joins:
  - name: supplier
    source: supplier_dim_mv
    on: "supplier.supplier_key = source.supplier_key"
  - name: part
    source: part_dim_mv
    on: "part.part_key = source.part_key"
  - name: lineitem
    source: samples.tpch.lineitem
    on: "lineitem.l_suppkey = source.supplier_key AND lineitem.l_partkey = source.part_key"
    cardinality: one_to_many
  - name: partsupp
    source: samples.tpch.partsupp
    on: "partsupp.ps_suppkey = source.supplier_key AND partsupp.ps_partkey = source.part_key"
    cardinality: one_to_many
dimensions:
  - name: supplier_name
    expr: supplier.supplier_name
    comment: "From supplier_dim_mv — NOT redeclared"
  - name: part_brand
    expr: part.brand
    comment: "From part_dim_mv — NOT redeclared"
measures:
  - name: qty_ordered
    expr: SUM(lineitem.l_quantity)
    display_name: "Qty Ordered"
  - name: qty_in_stock
    expr: SUM(partsupp.ps_availqty)
    display_name: "Qty In Stock"
  - name: supply_cost
    expr: SUM(partsupp.ps_supplycost)
    display_name: "Supply Cost"
$$

In [0]:
%sql
-- ============================================================================
-- WHY THE BRIDGE MATTERS: Fan-out proof
-- ============================================================================

-- CORRECT total_quantity (from lineitem alone):
SELECT 'Correct total (independent)' AS method, SUM(l_quantity) AS total_qty
FROM samples.tpch.lineitem
UNION ALL
-- WRONG: joining facts on supplier_key ONLY (single conformed dim, no bridge)
SELECT 'Fan-out (join on supplier only)', SUM(li.l_quantity)
FROM samples.tpch.lineitem li
JOIN samples.tpch.partsupp ps ON li.l_suppkey = ps.ps_suppkey
UNION ALL
-- CORRECT: joining facts through bridge on BOTH keys
SELECT 'Bridge (join on supplier+part)', SUM(li.l_quantity)
FROM samples.tpch.lineitem li
JOIN samples.tpch.partsupp ps ON li.l_suppkey = ps.ps_suppkey AND li.l_partkey = ps.ps_partkey

method,total_qty
Correct total (independent),765092358.00
Fan-out (join on supplier only),61207388640.00
Bridge (join on supplier+part),765092358.00


In [0]:
%sql
use catalog IDENTIFIER(:catalog);
use schema IDENTIFIER(:schema);

In [0]:
%sql
-- Business question: Where is demand outpacing supply?
SELECT nation_name,
       MEASURE(qty_ordered) AS qty_ordered,
       MEASURE(qty_in_stock) AS qty_in_stock,
       MEASURE(qty_ordered) / MEASURE(qty_in_stock) AS demand_to_stock_ratio
FROM orders_vs_supply_by_nation_mv
GROUP BY 1
ORDER BY demand_to_stock_ratio DESC
LIMIT 9;

nation_name,qty_ordered,qty_in_stock,demand_to_stock_ratio
KENYA,30554964.00,797164812,0.038329544330
PERU,30589229.00,798245254,0.038320589877
INDIA,31629388.00,825740578,0.038304267518
ARGENTINA,31379451.00,819235537,0.038303332293
JAPAN,30381286.00,793291625,0.038297752103
UNITED STATES,30458670.00,795375604,0.038294699821
JORDAN,29337131.00,766421034,0.038278086976
ETHIOPIA,30051814.00,785167373,0.038274404966
ROMANIA,30315360.00,792254052,0.038264695426
VIETNAM,30363265.00,793549776,0.038262584047


In [0]:
%sql
-- Business question: Who are our top suppliers in Brazil?
SELECT supplier_name,
       MEASURE(qty_ordered) AS units_sold,
       MEASURE(order_revenue) AS revenue
FROM home_dipankar_kushari.metric_view.orders_vs_supply_by_nation_mv
WHERE nation_name = 'BRAZIL'
GROUP BY 1
ORDER BY revenue DESC
LIMIT 9;

supplier_name,units_sold,revenue
Supplier#000004498,16772.00,27696101.4236
Supplier#000016996,16624.00,27573062.6326
Supplier#000045445,17039.00,27517245.5383
Supplier#000019959,16913.00,27439551.4143
Supplier#000023494,16480.00,27198081.1682
Supplier#000013996,16487.00,27196110.3255
Supplier#000016989,16632.00,27160169.9441
Supplier#000000992,16382.00,27097138.9724
Supplier#000031481,16525.00,27083051.0396
Supplier#000002473,16723.00,27030876.1940


In [0]:
%sql
-- Business question: Which product brands are most expensive to stock vs. what they earn?
SELECT part_brand,
       MEASURE(qty_ordered) AS units_ordered,
       MEASURE(supply_cost) AS total_supply_cost,
       MEASURE(qty_in_stock) AS units_in_stock
FROM orders_vs_supply_by_supplier_part_mv
GROUP BY 1
ORDER BY total_supply_cost DESC
LIMIT 9;

part_brand,units_ordered,total_supply_cost,units_in_stock
Brand#33,30968743.00,81054148.05,808600857
Brand#12,30882480.00,80867299.08,807890718
Brand#35,30817737.00,80821587.11,807565376
Brand#21,30837237.00,80687920.89,806003636
Brand#44,30809476.00,80657854.79,805361821
Brand#24,30705858.00,80287675.65,801023079
Brand#43,30638397.00,80209881.71,801456697
Brand#41,30588618.00,80172380.96,800558468
Brand#42,30585128.00,80121205.83,802109996
Brand#45,30570418.00,80086630.52,800141255


In [0]:
%sql
-- Business question: Supplier#000000042 — are they overstocked or understocked by brand?
SELECT supplier_name,
       part_brand,
       MEASURE(qty_ordered) AS demand,
       MEASURE(qty_in_stock) AS stock,
       MEASURE(qty_in_stock) - MEASURE(qty_ordered) AS surplus
FROM orders_vs_supply_by_supplier_part_mv
WHERE supplier_name = 'Supplier#000000042'
GROUP BY 1, 2
ORDER BY surplus ASC
LIMIT 9;

supplier_name,part_brand,demand,stock,surplus
Supplier#000000042,Brand#22,222.00,323,101.00
Supplier#000000042,Brand#44,195.00,1261,1066.00
Supplier#000000042,Brand#32,386.00,4836,4450.00
Supplier#000000042,Brand#55,228.00,9218,8990.00
Supplier#000000042,Brand#45,256.00,9881,9625.00
Supplier#000000042,Brand#15,426.00,13149,12723.00
Supplier#000000042,Brand#52,672.00,14614,13942.00
Supplier#000000042,Brand#35,425.00,14409,13984.00
Supplier#000000042,Brand#34,854.00,15173,14319.00
Supplier#000000042,Brand#54,1070.00,16155,15085.00


In [0]:
%sql
-- Business question: Where do we make the most money?
SELECT nation_name,
       MEASURE(order_revenue) AS revenue,
       MEASURE(qty_ordered) AS units
FROM orders_vs_supply_by_nation_mv
GROUP BY 1
ORDER BY revenue DESC
LIMIT 5

nation_name,revenue,units
INDIA,44978532738.6463,31629388.00
IRAQ,44718494962.4900,31424669.00
ARGENTINA,44620544239.5775,31379451.00
UNITED KINGDOM,44436882525.4700,31068546.00
INDONESIA,44425708259.1952,31219883.00


In [0]:
%sql
-- ============================================================================
-- SUMMARY
-- ============================================================================
--
-- All objects created in: :catalog.:schema (configurable via widgets)
--
-- ┌─────────────────────────────────────────────────────────────────────────┐
-- │  CONFORMED DIMENSION MVs (each owns ONLY its own attributes):           │
-- │    region_dim_mv     → region_key, region_name                          │
-- │    nation_dim_mv     → nation_key, nation_name, region_key (FK)         │
-- │    supplier_dim_mv   → supplier_key, supplier_name, nation_key (FK)     │
-- │    part_dim_mv       → part_key, part_name, brand, part_type            │
-- ├─────────────────────────────────────────────────────────────────────────┤
-- │  FACT MVs (measures + FKs ONLY):                                        │
-- │    lineitem_fact_mv  → l_suppkey, l_partkey + measures                  │
-- │    partsupp_fact_mv  → ps_suppkey, ps_partkey + measures                │
-- ├─────────────────────────────────────────────────────────────────────────┤
-- │  BRIDGE MV (for multiple conformed dimensions):                         │
-- │    supplier_part_bridge_dim_mv → (supplier_key, part_key) from data     │
-- ├─────────────────────────────────────────────────────────────────────────┤
-- │  MULTI-FACT MVs (joining facts through conformed dimensions):           │
-- │    orders_vs_supply_by_nation_mv → single conformed dim (supplier)      │
-- │    orders_vs_supply_by_supplier_part_mv → bridge (supplier+part)        │
-- └─────────────────────────────────────────────────────────────────────────┘
--
-- PATTERNS:
--   1. No repeated columns — each dim owns ONLY its attributes
--   2. Bridge from actual data — for multi-key conformed dimensions
--   2b. DRY bridge — cross-join in SOURCE + re-join (commented, DBR 18.1+)
--   3. Single conformed dim — dim as source + one_to_many facts
--   3b. Bridge multi-fact — bridge as source + one_to_many facts on BOTH keys
--
-- WIDGET CONFIGURATION:
--   Change "catalog" and "schema" widgets at top to deploy anywhere.
-- ============================================================================

SELECT :catalog || '.' || :schema AS deployed_to,
       'All patterns demonstrated — change widgets to redeploy' AS note

deployed_to,note
home_dipankar_kushari.metric_view,All patterns demonstrated — change widgets to redeploy
